# Week 3a.1 — Short-term Memory

Our agents so far have amnesia. Every `invoke` starts from nothing: the model does not remember what you asked one cell ago. Today we fix that for a single conversation. Memory across conversations and sessions is a different problem, which we take up later in the course.

In [1]:
from dotenv import load_dotenv
import os
import logging

load_dotenv()
assert os.getenv("GOOGLE_API_KEY"), "No GOOGLE_API_KEY found."

# silence a noisy advisory warning from the Google SDK
logging.getLogger("google_genai.models").setLevel(logging.ERROR)

print("API key loaded")

API key loaded


## 0. Model Setup

Execute one of the following to define the model.

If using Ollama, you will need to start it first (simply open the chat UI and send a message):
- https://docs.langchain.com/oss/python/integrations/chat/ollama

Verify that Ollama is serving a model:
- http://localhost:11434/

In [2]:
from langchain_ollama import ChatOllama

model = ChatOllama(model="qwen3.5:4b", reasoning=False)

In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model(model="gpt-4.1-mini")

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")

## 1. The problem

In [4]:
response = model.invoke("Hello from Boston.")
print(response.text)

Hello! Welcome from **Boston**, the "Harmony of Old and New." 🌉🎶

Whether you're exploring historic sites like the Freedom Trail, enjoying a cup of coffee in the seaport, or navigating the academic hub of Cambridge, I'm happy to help. What's on your mind today?


In [5]:
response = model.invoke("Where am I located?")
print(response.text)

I don't have access to your real-time location or personal data unless you provide it explicitly in our conversation. Where are you?


The model has no idea. It is **stateless**: nothing carries over from one call to the next. What looked like a conversation in a chat app was never memory inside the model; the application was resending the history every time.

We already have the tool for this: a call can take a **list of messages**. That list is the memory.

That is all short-term memory is: the application rereads the entire conversation to the model on every single call. Nothing is stored inside the model.

## 3. Agents with threads

For agents, LangChain packages this pattern so you do not manage the list yourself. Give `create_agent` a **checkpointer** and it saves the conversation state after every call, filed under a `thread_id` you choose. Same thread, same memory; new thread, blank slate.

- https://docs.langchain.com/oss/python/langchain/short-term-memory

In [12]:
from langchain.tools import tool
import requests

@tool
def get_weather(city: str) -> str:
    """Get the weather data for the city provided as an argument"""

    data = requests.get(f"https://wttr.in/{city}?format=j1").json()
    return data["current_condition"][0]

In [13]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model=model,
    tools=[get_weather],
    system_prompt="You're a helpful assistant who answers users' questions concisely.",
    checkpointer=InMemorySaver(),
)

config = {"configurable": {"thread_id": "1"}}

In [14]:
from langchain.messages import HumanMessage

message = [HumanMessage(content="Hello from Boston!")]
result = agent.invoke({"messages": message}, config)
print(result["messages"][-1].text)

Hello! How can I help you today?


In [20]:
message2 = [HumanMessage(content="Where am I located?")]
result = agent.invoke({"messages": message2}, config)
print(result["messages"][-1].text)

You are in **Boston**, Massachusetts, USA.


In [21]:
result

{'messages': [HumanMessage(content='Hello from Boston!', additional_kwargs={}, response_metadata={}, id='e37c13f7-08c0-4c2d-b501-a8e2ab206a96'),
  AIMessage(content='Hello! How can I help you today?', additional_kwargs={}, response_metadata={'model': 'qwen3.5:4b', 'created_at': '2026-09-21T18:40:57.254901Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1339949666, 'load_duration': 1296791, 'prompt_eval_count': 293, 'prompt_eval_duration': 988156000, 'eval_count': 10, 'eval_duration': 347458000, 'logprobs': None, 'model_name': 'qwen3.5:4b', 'model_provider': 'ollama'}, id='lc_run--01a0c545-6a68-7193-8951-8c5850276b2f-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 293, 'output_tokens': 10, 'total_tokens': 303}),
  HumanMessage(content='Where am I located?', additional_kwargs={}, response_metadata={}, id='502f87b7-284c-4fdd-8385-575f74cbe078'),
  AIMessage(content="You are in **Boston**, Massachusetts, USA. It's a major city in the northeastern United 

In [22]:
message3 = [HumanMessage(content="What's the weather like?")]

result = agent.invoke({"messages": message3}, config)
print(result["messages"][-1].text)

It is currently **cloudy** in Boston with a temperature of **62°F (17°C)**. The conditions feel cooler at **55°F (13°C)** due to a north-east wind blowing at 13 mph. There is no precipitation right now, but the humidity is 64%.


In [23]:
result

{'messages': [HumanMessage(content='Hello from Boston!', additional_kwargs={}, response_metadata={}, id='e37c13f7-08c0-4c2d-b501-a8e2ab206a96'),
  AIMessage(content='Hello! How can I help you today?', additional_kwargs={}, response_metadata={'model': 'qwen3.5:4b', 'created_at': '2026-09-21T18:40:57.254901Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1339949666, 'load_duration': 1296791, 'prompt_eval_count': 293, 'prompt_eval_duration': 988156000, 'eval_count': 10, 'eval_duration': 347458000, 'logprobs': None, 'model_name': 'qwen3.5:4b', 'model_provider': 'ollama'}, id='lc_run--01a0c545-6a68-7193-8951-8c5850276b2f-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 293, 'output_tokens': 10, 'total_tokens': 303}),
  HumanMessage(content='Where am I located?', additional_kwargs={}, response_metadata={}, id='502f87b7-284c-4fdd-8385-575f74cbe078'),
  AIMessage(content="You are in **Boston**, Massachusetts, USA. It's a major city in the northeastern United 

Same question, different thread:

In [ ]:
#TODO: invoke the agent with the same follow-up question but a new
# thread_id (for example "jordan") and see what changes.


One agent, many independent conversations, one line of difference. This is exactly how a support desk serves many customers at once, which is Friday's project.

## 4. Memory has a cost

The history grows on every turn, and the model rereads all of it on every call. You pay for that twice: context windows are finite, and tokens cost money.

In [24]:
#result = agent.invoke({"messages": [HumanMessage(content="Suggest a training plan for this week.")]}, config)

print(len(result["messages"]), "messages in the thread")

12 messages in the thread


In [25]:
from langchain.messages import trim_messages

# Keep only the most recent messages. Here token_counter=len counts
# messages; production systems count actual tokens.
trimmed = trim_messages(
    result["messages"],
    strategy="last",
    token_counter=len,
    max_tokens=4,
)

for m in trimmed:
    print(type(m).__name__, '-', m.text[:60])

HumanMessage - Where am I located?
AIMessage - You are in **Boston**, Massachusetts, USA.
HumanMessage - What's the weather like?
AIMessage - It is currently **cloudy** in Boston with a temperature of *


Trimming is the bluntest instrument: whatever falls off the end is gone, even if it was the customer's name. Smarter strategies, such as summarizing old turns or moving facts to external storage, are exactly where the course goes next week.

## 5. A chat loop

With threads, a real chat interface is a few lines. Uncomment and run; type `quit` to stop.

In [26]:
while True:
    user = input("You: ")
    if user.lower() in {"quit", "exit"}:
        break
    result = agent.invoke({"messages": [HumanMessage(content=user)]}, config)
    print("Assistant:", result["messages"][-1].text)

Assistant: Hello! How can I help you today? Would you like to check the weather in New York?
Assistant: It's currently **overcast** in New York, with a temperature of **72°F (22°C)**. The conditions feel slightly cooler at **66°F (19°C)** due to an east-north-east wind blowing at 10 mph. There is no precipitation right now, and the humidity is 41%.
Assistant: You are in **New York**, USA. Would you like more details about New York City?
Assistant: I wasn't able to fetch the weather forecast for tomorrow, so I can only provide the current conditions for New York. It is currently **overcast** with a temperature of **72°F (22°C)**, feeling like **66°F (19°C)** due to an east-north-east wind at 10 mph. The humidity is 41% and there is no precipitation. If you need the forecast for tomorrow, I would need access to specific forecasting tools.


## 6. A chat window

The terminal loop above works, but one import gives us a real chat interface. `gradio_ui.py` lives next to this notebook: it wraps any agent built with `create_agent`, shows tool calls as collapsible entries while the agent works, and sends the same `thread_id` on every turn, so the agent's own checkpointer does the remembering.

Run the cell and open the local URL it prints. Interrupt the kernel (or restart it) to stop the server.

In [ ]:
from gradio_ui import GradioUI

app = GradioUI(agent, {"configurable": {"thread_id": "gradio-demo"}}, title="Agent with Memory")
app.launch()